# Nixia GPU Training (Kaggle / Google Colab)

Notebook ini menjalankan build dataset, tokenizer, training GPU/CPU, eval, dan prompt eval.

Backend yang disarankan:
- Colab/Kaggle NVIDIA: `cuda` + Cargo feature `cuda-backend`.
- AMD/Intel lokal: `wgpu` + Cargo feature `wgpu-backend`.
- CPU fallback: `flex` tanpa feature GPU.


In [ ]:
# Runtime knobs. Adjust these before running all cells.
BACKEND = 'cuda'      # 'cuda', 'wgpu', or 'flex'
DEVICE_INDEX = 0
GPU_KIND = 'discrete' # wgpu only: default, discrete, integrated, virtual, cpu
BATCH_SIZE = 16       # increase until VRAM is almost full but stable
NUM_WORKERS = 2       # CPU dataloader workers; try 2-4 on hosted notebooks
EPOCHS = 20
SEQ_LEN = 128
LAYERS = 8
LR = '0.00005'
VOCAB_SIZE = 6000
ARTIFACTS = 'artifacts/nixia-gpu-run'
VOCAB = 'artifacts/vocab-gpu.txt'


In [ ]:
# Colab usually needs Rust. Kaggle often persists installs only for the session.
import os, shutil, subprocess, textwrap
if shutil.which('cargo') is None:
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
    os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version
!cargo --version


In [ ]:
# If this notebook is not already inside the repo, clone it here.
# Replace the URL with your own fork/private repo if needed.
import os
if not os.path.exists('Cargo.toml'):
    REPO_URL = 'https://github.com/your-user/nixia.git'
    !git clone {REPO_URL} nixia
    os.chdir('nixia')
print(os.getcwd())


In [ ]:
# GPU visibility check. NVIDIA should show nvidia-smi on Colab/Kaggle.
!nvidia-smi || true


In [ ]:
# Build and audit curated chat-clean corpus from checked-in templates.
!python tools/build_dataset.py --sources nixia_seed --max-rows-per-source 0 --synthesize 0 --valid-ratio 0.1 --min-score 0.8 --offline --extra-glob 'data/templates/nixia_dataset_*.txt' --output data/curated/chatclean_train.txt --valid-output data/curated/chatclean_valid.txt --report data/curated/chatclean_report.json
!python tools/audit_dataset.py --train data/curated/chatclean_train.txt --valid data/curated/chatclean_valid.txt --build-report data/curated/chatclean_report.json --json-output data/curated/chatclean_audit.json


In [ ]:
# Select Cargo features for the requested backend.
feature_map = {'cuda': 'cuda-backend', 'wgpu': 'wgpu-backend', 'flex': ''}
FEATURES = feature_map[BACKEND]
FEATURE_ARG = f'--features {FEATURES}' if FEATURES else ''
BACKEND_ARG = f'--backend {BACKEND}'
DEVICE_ARG = f'--device-index {DEVICE_INDEX}'
GPU_KIND_ARG = f'--gpu-kind {GPU_KIND}' if BACKEND == 'wgpu' else ''
print(FEATURE_ARG, BACKEND_ARG, DEVICE_ARG, GPU_KIND_ARG)


In [ ]:
# Train tokenizer. Re-run when corpus changes.
!cargo run --release -- tokenizer --corpus data/curated/chatclean_train.txt --vocab {VOCAB} --vocab-size {VOCAB_SIZE}


In [ ]:
# Long training. For high GPU utilization, increase BATCH_SIZE gradually.
cmd = f'''cargo run --release {FEATURE_ARG} -- train {BACKEND_ARG} {GPU_KIND_ARG} {DEVICE_ARG} \
  --preset nixia-micro --seq-len {SEQ_LEN} --layers {LAYERS} \
  --corpus data/curated/chatclean_train.txt --valid data/curated/chatclean_valid.txt \
  --vocab {VOCAB} --artifacts {ARTIFACTS} \
  --epochs {EPOCHS} --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS} --lr {LR}'''
print(cmd)
subprocess.run(cmd, shell=True, check=True)


In [ ]:
# Evaluate validation loss/perplexity. Eval currently uses CPU/Flex runtime for portability.
!cargo run --release -- eval --corpus data/curated/chatclean_valid.txt --vocab {VOCAB} --artifacts {ARTIFACTS}


In [ ]:
# Prompt regression eval and one manual generation sample.
!python tools/eval_prompts.py --artifacts {ARTIFACTS} --vocab {VOCAB} --output data/curated/prompt_eval_gpu.md
!cargo run --release -- generate --chat --artifacts {ARTIFACTS} --vocab {VOCAB} --prompt 'aku capek banget hari ini, rasanya pengen ditemenin ngobrol' --tokens 80 --temperature 0.55 --top-k 15 --top-p 0.85 --min-p 0.05
